In [ ]:
import re
from collections import Counter
from pathlib import Path

import pandas as pd

data_dir = Path("../data/raw")

AllClinicalXX – combines all the data from AccelerometryXX,
BiomarkersXX, JointSxXX, MedHistXX, NutritionXX, PhysExamXX, and
SubjectCharXX.

In [2]:
df = pd.read_csv(data_dir / "AllClinical00.csv")

In [3]:
print(df.shape)

n_rows, n_cols = df.shape

(4796, 1187)


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4796 entries, 0 to 4795
Columns: 1187 entries, ID to V00WORKAMT
dtypes: float64(251), int64(3), str(933)
memory usage: 43.4 MB


In [5]:
prefixes = [col[:3] for col in df.columns]

Counter(prefixes).most_common()

[('V00', 905), ('P01', 233), ('P02', 47), ('ID', 1), ('VER', 1)]

V00 Baseline: Enrollment

P01 Baseline: Screening

P02 Baseline: Initial Eligibility

In [6]:
for prefix in ['V00', 'P01', 'P02']:
    cols = [c for c in df.columns if c.startswith(prefix)]

    print(f"\n{prefix}: {len(cols)} variables")
    print(cols)


V00: 905 variables
['V00BLDCOLL', 'V00BLDHRS1', 'V00BLDHRS2', 'V00BLDRAW1', 'V00BLDRAW2', 'V00BLSURD1', 'V00BLSURD2', 'V00CITRATE', 'V00EDTA', 'V00excess1', 'V00excess2', 'V00hemat1', 'V00hemat2', 'V00hoursp1', 'V00hoursp2', 'V00hrsuc1', 'V00hrsuc2', 'V00illpwk1', 'V00illpwk2', 'V00LEAKAG1', 'V00LEAKAG2', 'V00MRSEQNL', 'V00MRSEQNR', 'V00MULTST1', 'V00MULTST2', 'V00othvp1', 'V00othvp2', 'V00pdate1', 'V00pdate2', 'V00PLAQHR1', 'V00PLAQHR2', 'V00qovp1', 'V00qovp2', 'V00SEAQHR1', 'V00SEAQHR2', 'V00SERUM', 'V00ucdate1', 'V00ucdate2', 'V00URINHR1', 'V00URINHR2', 'V00URINOB1', 'V00URINOB2', 'V00URNCOLL', 'V00URSURD1', 'V00URSURD2', 'V00vcoll1', 'V00vcoll2', 'V00vein1', 'V00vein2', 'V00void1', 'V00void2', 'V00SF1', 'V00SF2', 'V00SF3', 'V00SF4', 'V00SF5', 'V00SF6', 'V00SF7', 'V00SF8', 'V00SF9', 'V00SF10', 'V00SF11', 'V00SF12', 'V00WPRKN1', 'V00WPRKN2', 'V00WPRKN3', 'V00WPRKN4', 'V00WPRKN5', 'V00KPRKN1', 'V00KPRKN2', 'V00KPRKN3', 'V00P7RKFR', 'V00WSRKN1', 'V00WSRKN2', 'V00KSXRKN1', 'V00KSXRKN2'

In [13]:
nutrition = pd.read_csv(data_dir / "oai_nutrition01_definitions.csv")
medical_history = pd.read_csv(data_dir / "oai_oarisk01_definitions.csv")
biomarkers = pd.read_csv(data_dir / "oai_labcollection01_definitions.csv")
comorbidity = pd.read_csv(data_dir / "oai_charlson01_definitions.csv")
physical = pd.read_csv(data_dir / "oai_physfunct01_definitions.csv")
depression = pd.read_csv(data_dir / "oai_ces_d01_definitions.csv")
koos = pd.read_csv(data_dir / "oai_koos_womac01_definitions.csv")
pain = pd.read_csv(data_dir / "oai_oapain01_definitions.csv")
pase = pd.read_csv(data_dir / "oai_pase01_definitions.csv")
sf_12 = pd.read_csv(data_dir / "oai_sf1201_definitions.csv")

In [14]:
columns = df.columns.tolist()

def remove_prefix(column):
    return re.sub(r'^[A-Z]\d{2}', '', column)

variables = pd.DataFrame({
    "AllClinical00": columns,
    "ElementName_candidate": [remove_prefix(c).lower() for c in columns]
})

print(variables.head())

  AllClinical00 ElementName_candidate
0            ID                    id
1       VERSION               version
2    V00BLDCOLL               bldcoll
3    V00BLDHRS1               bldhrs1
4    V00BLDHRS2               bldhrs2


In [10]:
def add_dataset_match(df, dictionary, dataset_name):
    mask = df['ElementName_candidate'].isin(dictionary['ElementName'])

    df.loc[mask, 'Dataset'] = dataset_name

    lookup = dictionary.set_index('ElementName')

    mask = df['ElementName_candidate'].isin(lookup.index)

    df.loc[mask, 'Dataset'] = dataset_name

    dictionary_fields = [
        col for col in dictionary.columns
        if col != 'ElementName'
    ]

    for field in dictionary_fields:
        if field not in df.columns:
            df[field] = None

        df.loc[mask, field] = (
            df.loc[mask, 'ElementName_candidate']
            .map(lookup[field])
        )

        
    return df

Koos/Womac data dictionary did not match AllClinical; manual mapping to then replace names

In [15]:
koos_crosswalk = pd.read_csv(data_dir / "koos_womac_crosswalk.csv")

misc_crosswalk = pd.read_csv(data_dir / "misc_crosswalk.csv")

# Remove white spaces
misc_crosswalk["AllClinical00"] = (
    misc_crosswalk["AllClinical00"].astype("string").str.strip()
)

all_crosswalk = pd.concat(
    [koos_crosswalk, misc_crosswalk],
    ignore_index=True
)


crosswalk_lookup = dict(
    zip(
        all_crosswalk["AllClinical00"],
        all_crosswalk["ElementName"]
    )
)

variables["ElementName_candidate"] = (
    variables["AllClinical00"]
    .map(crosswalk_lookup)
    .fillna(variables["ElementName_candidate"])
)


In [16]:
variables = add_dataset_match(variables, nutrition, 'Nutrition')
variables = add_dataset_match(variables, medical_history, 'Medical History')
variables = add_dataset_match(variables, biomarkers, 'Biomarkers')
variables = add_dataset_match(variables, comorbidity, 'Comorbidity')
variables = add_dataset_match(variables, physical, 'Physical Function')
variables = add_dataset_match(variables, depression, 'Depression')
variables = add_dataset_match(variables, koos, 'KOOS/WOMAC')
variables = add_dataset_match(variables, pain, 'Pain and Medication')
variables = add_dataset_match(variables, pase, 'Physical Activity')
variables = add_dataset_match(variables, sf_12, 'Medical Outcomes')

In [17]:
variables['Dataset'].value_counts(dropna=False)

Dataset
Medical History        309
Nutrition              285
Pain and Medication    197
Physical Function      170
KOOS/WOMAC              89
Biomarkers              49
Comorbidity             28
Depression              21
Physical Activity       19
Medical Outcomes        18
NaN                      2
Name: count, dtype: int64

In [18]:
variables.head()

,AllClinical00,ElementName_candidate,Dataset,DataType,Size,Required,ElementDescription,ValueRange,Notes,Aliases
0,ID,id,NaN,None,None,None,None,None,None,None
1,VERSION,version,Medical Outcomes,String,20.0,Recommended,Version/code of assessment,NaN,"e.g., Self-Report, Parent Form, Short Form, Re...",NaN
2,V00BLDCOLL,bldcoll,Biomarkers,Integer,NaN,Recommended,Phlebotomy: which draw(s) blood obtained at,0::3,0= No sample collected at visit; 1= Sample at ...,NaN
3,V00BLDHRS1,bldhrs1,Biomarkers,String,20.0,Recommended,Phlebotomy: time venipuncture completed (first...,NaN,NaN,NaN
4,V00BLDHRS2,bldhrs2,Biomarkers,String,20.0,Recommended,Phlebotomy: time venipuncture completed (repea...,NaN,NaN,NaN


In [19]:
manual_info = {
    "ID": {
        "ElementName_candidate": "id",
        "Dataset": "Participant Identifier",
        "ElementDescription": "Release ID",
    },

    "V00HANDED": {
        "ElementName_candidate": "handed",
        "Dataset": "Participant Characteristics",
        "DataType": "Integer",
        "ElementDescription": "Dominant hand for x-ray",
        "ValueRange": "0::3",
        "Notes": "1= Right handed; 2= Left handed; 3= Ambidexterous"
    },
}

for variable, info in manual_info.items():
    mask = variables["AllClinical00"] == variable
    
    for column, value in info.items():
        if column not in variables.columns:
            variables[column] = pd.NA
        
        variables.loc[mask, column] = value

In [20]:
variables.to_csv(Path("../data/processed") / "AllClinical00_variable_inventory.csv", index=False)

In [21]:
variables.head()

,AllClinical00,ElementName_candidate,Dataset,DataType,Size,Required,ElementDescription,ValueRange,Notes,Aliases
0,ID,id,Participant Identifier,None,None,None,Release ID,None,None,None
1,VERSION,version,Medical Outcomes,String,20.0,Recommended,Version/code of assessment,NaN,"e.g., Self-Report, Parent Form, Short Form, Re...",NaN
2,V00BLDCOLL,bldcoll,Biomarkers,Integer,NaN,Recommended,Phlebotomy: which draw(s) blood obtained at,0::3,0= No sample collected at visit; 1= Sample at ...,NaN
3,V00BLDHRS1,bldhrs1,Biomarkers,String,20.0,Recommended,Phlebotomy: time venipuncture completed (first...,NaN,NaN,NaN
4,V00BLDHRS2,bldhrs2,Biomarkers,String,20.0,Recommended,Phlebotomy: time venipuncture completed (repea...,NaN,NaN,NaN


In [32]:
df.head(2)

,ID,VERSION,V00BLDCOLL,V00BLDHRS1,V00BLDHRS2,V00BLDRAW1,V00BLDRAW2,V00BLSURD1,V00BLSURD2,V00CITRATE,...,V00PASE4,V00PASE4HR,V00PASE5,V00PASE5HR,V00PASE6,V00PASE6HR,V00WEEKWK,V00WKHR7CV,V00WORK7,V00WORKAMT
0,9000099,0.2.3,1: Sample at 1st collection only,35400.0,NaN,1: Yes,.: Missing Form/Incomplete Workbook,NaN,NaN,1: Sample at 1st collection only,...,0: Never,.: Missing Form/Incomplete Workbook,3: Often (5-7 days),1: Less than 1 hour,2: Sometimes (3-4 days),1: Less than 1 hour,52.0,50.0,1: Yes,1: Sitting
1,9000296,0.2.3,1: Sample at 1st collection only,30780.0,NaN,1: Yes,.: Missing Form/Incomplete Workbook,NaN,NaN,1: Sample at 1st collection only,...,0: Never,.: Missing Form/Incomplete Workbook,0: Never,.: Missing Form/Incomplete Workbook,0: Never,.: Missing Form/Incomplete Workbook,52.0,50.0,1: Yes,2: Sitting/standing/walking


# Check empty in Allclinical

In [36]:
blank   = df.isna()
coded   = df.astype(str).apply(lambda s: s.str.startswith(".:"))
missing = blank | coded

print("Columns with most missing values:\n",missing.sum().sort_values(ascending=False).head(20) )         # cols with most missing values
print("\nPercentage of missing values per column:",(missing.mean() * 100).round(1))                              # % missing per column
print("\nMissing count per participant:",missing.sum(axis=1))                                          # missing count per participant

Columns with most missing values:
 P02CNC14      4796
V00REASW8     4796
V00REASW15    4796
P02HR11       4796
P02CMDK       4796
V00DKP400W    4796
V00RFP400W    4796
V00FFQ101     4796
V00CHELCUR    4796
V00CHELNUM    4796
V00FOLKNUM    4796
V00FOLKCUR    4796
V00URSURD2    4796
P01LRR3       4796
P01LRL2       4796
P01BPDK       4796
P01LRL3       4796
P01HRSLDK     4796
P01HPNLDK     4796
P01HRSRDK     4796
dtype: int64

Percentage of missing values per column: ID             0.0
VERSION        0.0
V00BLDCOLL     0.0
V00BLDHRS1     0.4
V00BLDHRS2    99.4
              ... 
V00PASE6HR    56.7
V00WEEKWK     37.9
V00WKHR7CV     0.1
V00WORK7       0.1
V00WORKAMT    30.4
Length: 1187, dtype: float64

Missing count per participant: 0       351
1       332
2       325
3       306
4       320
       ... 
4791    330
4792    350
4793    359
4794    339
4795    326
Length: 4796, dtype: int64
